# Caso MPG

Breve descripción del dataset 'MPG' y sus columnas:

El conjunto de datos MPG contiene registros de automóviles con medidas relacionadas con el consumo de combustible y características del vehículo. Es útil para explorar cómo las propiedades del vehículo afectan la eficiencia de combustible.

Columnas principales (explicación muy simple):
- **mpg:** consumo de combustible (millas por galón).
- **cylinders:** número de cilindros del motor.
- **displacement:** tamaño del motor (desplazamiento).
- **horsepower:** potencia del motor (caballos de fuerza).
- **weight:** peso del vehículo.
- **acceleration:** medida de aceleración del vehículo.
- **model_year:** año del modelo del vehículo.
- **origin:** región de origen del vehículo (por ejemplo: USA, Europe, Japan).
- **name:** nombre o modelo del vehículo.

Uso sugerido: usa estas columnas para predecir 'mpg' y para análisis exploratorio sencillo (correlaciones y visualizaciones).

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

## Extracción de Datos

In [ ]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/mpg.db"
r = requests.get(url)

with open("mpg.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("mpg.db")

query = """
SELECT
    O.mpg,
    O.cylinders,
    O.displacement,
    O.horsepower,
    O.weight,
    O.acceleration,
    O.model_year,
    ORG.origin,
    N.name
FROM
    Observation AS O
JOIN
    Origin AS ORG ON O.origin_id = ORG.origin_id
JOIN
    Name AS N ON O.name_id = N.name_id
"""

df = pd.read_sql_query(query, conn)
df.head()

## Exploración de Datos

In [ ]:
# Identificar tipos de datos

In [ ]:
# Detectar nulos
# ¿Qué variables tienen nulos?

In [ ]:
# tratar nulos
# Si representan más del 5% imputar valores, de lo contrario eliminar
# Ojo (pandas 3): usa df['col'] = df['col'].fillna(valor), NO df['col'].fillna(valor, inplace=True)

In [ ]:
# Se debe(n) de conservar la(s) variable(s) con nulos?
# Hacer lo correcto.

In [ ]:
# Resumen; estadísticas descriptivas del dataset

### Distribuciones Numéricas

In [ ]:
# Visualización de distribuciones para variables numéricas
# 1. Crea la variable numerical_vars -> tipo lista
# 2. Llama a plot_distributions(df, numerical_vars)


### Frecuencias Categóricas

In [ ]:
# Visualización de frecuencias de variables categóricas
# 1. Crea la variable categorical_vars -> tipo lista
# 2. Llama a plot_frequencies(df, categorical_vars, top_n=20)
#    ('name' tiene cientos de valores: top_n=20 muestra sólo los más comunes)


### Matriz de Correlación

In [ ]:
# Matriz de correlación de las variables numéricas
# Llama a plot_correlation_matrix(df, numerical_vars)


In [ ]:
# Pairplot de las variables numéricas, coloreado por origen
# Este va de regalo
plot_pairplot(df, numerical_vars, color='origin')

### Conclusiones del analisis exploratorio
RELLENAR CON TUS COMENTARIOS

## Preprocesamiento

In [ ]:
# Identificar mis variables independientes (X) y la dependiente (y)
# NO INCLUYAN LA VARIABLE 'name'

In [ ]:
# Hacer train/test split del 80/20 con random_state=42

from sklearn.model_selection import train_test_split

In [ ]:
# One Hot Encoder a 'origin'
# Aquí deben de hacer fit_transform() en X_train['origin']
# Y hacer transform() unicamente en X_test['origin']

from sklearn.preprocessing import OneHotEncoder, TargetEncoder

In [ ]:
# Combinar variable categórica 'origin' con los datasets originales

## Ajustar Modelo

In [ ]:
# Entrenar y evaluar el modelo de regresión lineal
import statsmodels.api as sm

# Primero probar un caso simple
# mpg y su variable más correlacionada



# Agregar una constante para el término de intersección

# Ajustar el modelo con statsmodels

# Imprimir el resumen datos de entrenamiento

In [ ]:
# Calcular R² y RMSE en test con statsmodels
# Hay que agregar la constante al test también, igual que en train

from sklearn.metrics import mean_squared_error, r2_score
from math import sqrt

In [ ]:
# Visualizar la recta ajustada (datos de entrenamiento)
# Llama a plot_simple_regression(X_train_<variable>, y_train, results)


In [ ]:
# Ahora con TODAS las variables
import statsmodels.api as sm

# Agregar una constante para el término de intersección

# Ajustar el modelo con statsmodels

# Imprimir el resumen datos de entrenamiento

In [ ]:
# Calcular R² y RMSE en test con statsmodels
# Hay que agregar la constante al test también, igual que en train

In [ ]:
# ¿Qué tan cerca están las predicciones de los valores reales?
# Llama a plot_actual_vs_predicted(y_test, y_pred) y a plot_residuals(y_test, y_pred)


💡 En Machine Learning puro se prioriza R². En econometría y ciencias sociales se prioriza la validez estadística. Ambos mundos son válidos, pero tienen distintos criterios de éxito.